In [1]:
from sqlalchemy import (
    Boolean,
    Column,
    Float,
    Integer,
    MetaData,
    String,
    Table,
    text,
    update,
)

from egomimic.utils.aws.aws_sql import (
    TableRow,
    create_default_engine,
    episode_table_to_df,
)

In [2]:
engine = create_default_engine()

Tables in schema 'app': ['episodes']


In [3]:
df = episode_table_to_df(engine)
df

,episode_hash,operator,task,embodiment,robot_name,num_frames,task_description,scene,objects,zarr_processed_path,zarr_mp4_path,zarr_processing_error,is_deleted,data_type
0,00022b62-0a07-4ac7-bb74-9b4da47f6c6e,,sort the stationery into containers,eva_bimanual,eva_bimanual,1008,sort the stationery into containers,,,/workspace/eva/abc130k_zarr/00022b62-0a07-4ac7...,,,False,robot
1,0007e628-5799-4943-a005-859a8295126f,,roll the ties,eva_bimanual,eva_bimanual,2898,roll the ties,,,/workspace/eva/abc130k_zarr/0007e628-5799-4943...,,,False,robot
2,0008aa90-9957-4ca7-bf85-5b5cb8c26811,,fill the litter box with clean litter,eva_bimanual,eva_bimanual,7313,fill the litter box with clean litter,,,/workspace/eva/abc130k_zarr/0008aa90-9957-4ca7...,,,False,robot
3,000c166c-a109-4bc3-a966-8236de22f2f2,,put the credit cards into the card holder,eva_bimanual,eva_bimanual,4678,put the credit cards into the card holder,,,/workspace/eva/abc130k_zarr/000c166c-a109-4bc3...,,,False,robot
4,0018ef0c-2db7-4965-84d2-2e5796d24b06,,"fold the napkin, place the utensils inside, an...",eva_bimanual,eva_bimanual,3392,"fold the napkin, place the utensils inside, an...",,,/workspace/eva/abc130k_zarr/0018ef0c-2db7-4965...,,,False,robot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260268,ffe7bd38-a4fe-488e-9968-5573fcad61e3,,sort the stationery into containers,eva_bimanual,eva_bimanual,2583,sort the stationery into containers,,,/workspace/eva/abc130k_zarr/ffe7bd38-a4fe-488e...,,,False,robot
260269,ffe7fd1b-3b05-49a0-b389-1510219cb024,,fold and stack the tank tops,eva_bimanual,eva_bimanual,772,fold and stack the tank tops,,,/workspace/eva/abc130k_zarr/ffe7fd1b-3b05-49a0...,,,False,robot
260270,ffebb6fc-09f2-4618-9340-f22021aaadea,,insert the plug into the switch port,eva_bimanual,eva_bimanual,6185,insert the plug into the switch port,,,/workspace/eva/abc130k_zarr/ffebb6fc-09f2-4618...,,,False,robot
260271,ffed5dbb-d0b6-4c8d-8c20-fd2904eebd8d,,"open, take out, and arrange the luggage",eva_bimanual,eva_bimanual,8977,"open, take out, and arrange the luggage",,,/workspace/eva/abc130k_zarr/ffed5dbb-d0b6-4c8d...,,,False,robot


In [32]:
# Select tasks closest to target_task_hours, up to the number given by total_hours_budget / target_task_hours
import numpy as np

# User options
total_hours_budget = 240  # <-- set your total hours budget here
target_task_hours = 24    # <-- desired hours per task
tolerance = 2             # <-- plus/minus tolerance for per-task hours

# Compute total frames per task
frames_per_task = df.groupby("task")["num_frames"].sum()

# Convert frames to hours (frames / 30 FPS / 3600 seconds)
hours_per_task = frames_per_task / 30 / 3600

# Find tasks where hours is within the specified range
mask = ((hours_per_task >= (target_task_hours - tolerance)) & 
        (hours_per_task <= (target_task_hours + tolerance)))
tasks_in_range = hours_per_task[mask]

# Compute the distance from target for each task
distance_from_target = (tasks_in_range - target_task_hours).abs()

# How many tasks to pick?
num_tasks = int(total_hours_budget // target_task_hours)

# Order by closest to target
tasks_closest = distance_from_target.sort_values().head(num_tasks)
selected_task_names = list(tasks_closest.index)
selected_tasks = tasks_in_range.loc[selected_task_names]

print(f"Total hours budget: {total_hours_budget}")
print(f"Target hours per task: {target_task_hours} (+/- {tolerance})")
print(f"Number of top closest tasks to select: {num_tasks}")
print("Top selected tasks closest to target (task name and hours):")
print(selected_tasks)
print("Selected task names:", selected_task_names)
print("Actual sum hours:", selected_tasks.sum())

Total hours budget: 240
Target hours per task: 24 (+/- 2)
Number of top closest tasks to select: 10
Top selected tasks closest to target (task name and hours):
task
packaging_clothes                                               23.711583
insert the wireless bluetooth earbuds into the charging case    24.475417
assembling_boxes                                                23.372213
repairing_electronics                                           25.643250
wrapping_bouquets                                               25.780213
peeling_garlic                                                  25.876954
Name: num_frames, dtype: float64
Selected task names: ['packaging_clothes', 'insert the wireless bluetooth earbuds into the charging case', 'assembling_boxes', 'repairing_electronics', 'wrapping_bouquets', 'peeling_garlic']
Actual sum hours: 148.85962962962964


In [7]:
# Flagship Fold Clothes Data
print("Number of unique tasks:", df["robot_name"].unique())

Number of unique tasks: <ArrowStringArray>
['eva_bimanual', 'mecka_bimanual']
Length: 2, dtype: str


## Data Diversity Experiment — 5 nested levels, fixed 240h budget, deduped tasks

Human pretraining data only (`data_type == 'flagship'`). Two fixes over naive task selection:

1. **Exact budgets** — instead of picking tasks whose natural totals are *near* the per-task
   target (which runs out of tasks above ~18h/task), pick tasks with **at least** the target
   hours and subsample episodes down to the exact per-task budget.
2. **Semantic dedup** — the table is full of near-duplicate names (`packing_snacks` /
   `packaging_snacks`, `dishwashing` / `washing_dishes`, the dough-ball cluster, …).
   `TASK_FAMILIES` groups them; only the family **representative** (most-hours member) enters
   the candidate pool, so nominal task count = real semantic diversity. Training data is drawn
   from the representative's episodes only (one clean language label per family).

Task sets are nested (top-N families by available hours), so the only axis that changes
between levels is diversity.

| Level | Tasks | Hours/task | Total |
|-------|-------|-----------|-------|
| D1    | 5     | 48        | 240   |
| D2    | 10    | 24        | 240   |
| D3    | 20    | 12        | 240   |
| D4    | 40    | 6         | 240   |
| D5    | 80    | 3         | 240   |

In [ ]:
# Curated near-duplicate task groups (flagship tasks >= 3h). Family representative =
# the member with the most hours; absorbed members are excluded from the candidate pool.
TASK_FAMILIES = [
    # dishes / cutlery
    ["dishwashing", "washing_dishes", "rinsing_dishes", "drying_dishes", "wiping_plates", "wiping_bowls"],
    ["cleaning_cutlery", "wiping_cutlery", "wiping_spoons", "cleaning_spoons", "polishing_cutlery"],
    ["wrapping_cutlery", "packing_cutlery", "packaging_cutlery", "packing_cutlery_sets", "organizing_cutlery", "folding_napkins"],
    # clothes / fabric
    ["folding_clothes", "folding_towels"],
    ["ironing_clothes", "ironing_and_folding_clothes"],
    ["cleaning_clothes", "washing_clothes"],
    ["packaging_clothes", "packing_clothes", "packaging_socks"],
    ["sewing_clothes", "sewing_fabrics", "sewing_fabric", "sewing"],
    ["cutting_fabric", "cutting_fabrics"],
    # shoes
    ["cleaning_shoes", "brushing_shoes", "scrubbing_shoes"],
    # dough
    ["shaping_dough", "shaping_dough_balls", "rolling_dough_balls", "making_dough_balls",
     "preparing_dough_balls", "twisting_dough", "arranging_dough"],
    ["rolling_dough", "flattening_dough"],
    ["kneading_dough", "mixing_dough", "preparing_dough"],
    ["cutting_dough", "portioning_dough", "dividing_dough", "weighing_dough"],
    # baking / pastry
    ["shaping_pastries", "making_pastries", "preparing_pastries", "making_cookies", "shaping_cookies"],
    ["decorating_pastries", "decorating_cakes", "piping_dough"],
    ["packaging_pastries", "packing_pastries", "packaging_cookies", "packing_cookies", "packaging_bread"],
    ["making_dumplings", "making_spring_rolls"],
    # flowers
    ["making_flowers", "assembling_flowers", "crafting_flowers", "making_artificial_flowers",
     "assembling_artificial_flowers", "making_paper_flowers", "making_ribbon_flowers",
     "making_flower_petals", "shaping_petals", "making_ribbon_petals"],
    ["arranging_flowers", "making_bouquets", "assembling_bouquets"],
    ["wrapping_flowers", "wrapping_bouquets", "wrapping_stems"],
    # plants / garden
    ["potting_plants", "filling_pots", "mixing_soil", "planting_seedlings", "planting_cuttings"],
    ["weeding_plants", "weeding", "clearing_weeds"],
    ["flattening_leaves", "processing_leaves"],
    # packaging
    ["packaging_snacks", "packing_snacks", "arranging_snacks"],
    ["packaging_items", "packing_items", "packaging_goods"],
    ["packaging_food", "packaging_foods", "packing_food"],
    ["packaging_seeds", "packing_seeds", "bagging_seeds"],
    ["packaging_powders", "packaging_powder", "packing_powder", "scooping_powder"],
    ["packaging_grains", "packaging_soybeans"],
    ["packaging_nuts", "packaging_peanuts"],
    ["filling_bags", "sealing_bags"],
    ["filling_bottles", "filling_perfume_bottles"],
    ["wrapping_gifts", "packaging_gifts", "wrapping_boxes"],
    # boxes / paper
    ["folding_boxes", "assembling_boxes", "folding_pastry_boxes", "assembling_gift_boxes", "making_paper_bags"],
    ["folding_paper", "folding_papers"],
    ["cutting_paper", "cutting_labels", "cutting_stickers", "cutting_cardboard"],
    # food prep
    ["cooking", "frying_food", "preparing_food"],
    ["mixing_ingredients", "mixing_batter", "preparing_ingredients"],
    ["chopping_vegetables", "cutting_vegetables", "slicing_vegetables", "preparing_vegetables",
     "chopping_onions", "slicing_onions", "chopping_carrots", "chopping_garlic"],
    ["peeling_garlic", "peeling_vegetables", "peeling_potatoes", "peeling_onions", "peeling_coconuts", "peeling_eggs"],
    ["slicing_meat", "cutting_meat", "chopping_meat", "preparing_meat", "cutting_chicken"],
    ["skewering_meat", "skewering_food"],
    ["preparing_drinks", "filling_cups", "making_coffee", "preparing_beverages"],
    # repair / electronics
    ["repairing_phones", "disassembling_phones"],
    ["repairing_electronics", "soldering_electronics", "cleaning_motherboards"],
    ["repairing_laptops", "disassembling_laptops", "repairing_computers"],
    # cleaning
    ["cleaning_furniture", "cleaning_tables", "cleaning_surfaces", "cleaning_windows"],
    ["cleaning_kitchen", "cleaning_stoves", "cleaning_sink"],
    ["cleaning_trays", "cleaning_containers", "cleaning_bottles"],
    ["cleaning_bird_nests", "cleaning_birds_nests"],
    ["vacuuming", "sweeping", "mopping_floors"],
    # sorting / beads / ribbons / crafts
    ["sorting_beans", "sorting_coffee_beans", "sorting_beads", "sorting_seeds"],
    ["stringing_beads", "threading_beads"],
    ["tying_ribbons", "making_bows", "making_ribbon_bows", "folding_ribbons",
     "wrapping_ribbons", "cutting_ribbons", "preparing_ribbons"],
    ["assembling_decorations", "making_decorations", "making_crafts", "crafting_decorations"],
    ["making_jewelry", "polishing_jewelry"],
    ["shaping_pipe_cleaners", "twisting_pipe_cleaners"],
    # wood / misc
    ["woodworking", "planing_wood", "carving_wood"],
    ["sanding_furniture", "sanding_wood"],
    ["wrapping_sticks", "bundling_sticks"],
]

In [ ]:
# Diversity levels over deduped families: nested task sets, exact per-task hour budgets
TOTAL_BUDGET = 240            # total pretraining hours per experiment
LEVELS = [5, 10, 20, 40, 80]  # tasks per level (hours/task = TOTAL_BUDGET / n)
SEED = 42                     # episode subsampling seed

human = df[(df["data_type"] == "flagship") & (~df["is_deleted"])].copy()
human["hours"] = human["num_frames"] / 30 / 3600
hours_per_task = human.groupby("task")["hours"].sum()

# family representative = most-hours member; absorbed members leave the candidate pool
absorbed = set()
for group in TASK_FAMILIES:
    rep = hours_per_task.loc[group].idxmax()
    absorbed |= set(group) - {rep}
rep_hours = hours_per_task.drop(index=list(absorbed)).sort_values(ascending=False)
print(f"{len(hours_per_task)} task names -> {len(rep_hours)} deduped families\n")

diversity_levels = {}
for n in LEVELS:
    per_task_budget = TOTAL_BUDGET / n
    tasks = list(rep_hours.head(n).index)
    assert (rep_hours.head(n) >= per_task_budget).all(), (
        f"level {n}: some families have < {per_task_budget}h available"
    )

    # Subsample the representative's episodes: shuffle, greedily accumulate up to budget
    episode_hashes, actual_hours = [], 0.0
    for t in tasks:
        eps = human[human["task"] == t].sample(frac=1, random_state=SEED)
        kept = eps[eps["hours"].cumsum() <= per_task_budget]
        episode_hashes += list(kept["episode_hash"])
        actual_hours += kept["hours"].sum()

    diversity_levels[n] = {
        "hours_per_task": per_task_budget,
        "tasks": tasks,
        "episode_hashes": episode_hashes,
    }
    print(f"D{LEVELS.index(n)+1}: {n:>3} tasks x {per_task_budget:g}h "
          f"-> {len(episode_hashes)} episodes, {actual_hours:.1f}h actual")
    print("   tasks:", tasks, "\n")

# Nesting check: each level's task set contains the previous level's
for a, b in zip(LEVELS, LEVELS[1:]):
    assert set(diversity_levels[a]["tasks"]) <= set(diversity_levels[b]["tasks"])
print("All levels nested: D1 ⊂ D2 ⊂ D3 ⊂ D4 ⊂ D5")